# Comment Category Prediction

## Submission By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

## Import all the required libraries

In [ ]:
import gc
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from scipy.optimize import minimize
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import (
    KFold,
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MaxAbsScaler,
    OrdinalEncoder,
    PowerTransformer,
    RobustScaler,
)
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

## Config

In [ ]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("datasets/")

# Ensemble top K models
TOP_K = 3

# Random State for reproducibility
RANDOM_STATE = 42

# Controls HPT
PERFORM_TUNING = False

# Cross-validation splits
CV_SPLITS = 3

# Controls XG Boost (can be slow to train)
ENABLE_XGBOOST = False

# Dimensionality reduction for text features
ENABLE_SVD = False
SVD_N_COMPONENTS = 500

## Data loading

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

In [ ]:
for _name, _df in [
    ("train", train),
    ("test", test),
    ("sample_sub", sample_sub),
]:
    print(f" {_name} ".center(50, "="))
    print("\nDataset Info:")
    print(_df.info())
    print("\nDataset Description:")
    print(_df.describe(include="all"))
    # print("\nTable Head (10 rows):")
    # print(_df.head(10))
    print("\n\n")

## Helper Functions

In [ ]:
def header(msg: str) -> None:
    """
    Print a formatted header message.
    :param msg: Message to be printed as header.
    :return:
    """
    print("\n" + "=" * 50)
    print(msg.center(50))
    print("=" * 50)

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean the input text by removing URLs, special characters, and extra spaces.
    :param text: Input text string.
    :return: Cleaned text string.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def calc_sparsity(matrix) -> float:
    """
    Calculate the sparsity of a given matrix.
    :param matrix: Input matrix (numpy array or sparse matrix).
    :return: Sparsity in percentage.
    """
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero_elements = matrix.nnz if issparse(matrix) else np.count_nonzero(matrix)
    sparsity = 1.0 - (non_zero_elements / total_elements)
    return sparsity * 100

## Exploratory Data Analysis

In [ ]:
train.shape, test.shape, sample_sub.shape

In [ ]:
train.sample(10)

In [ ]:
# Label distribution
print(train["label"].value_counts())
print(f"\nProportions:\n{train['label'].value_counts(normalize=True)}")

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=train, x="label")
plt.title("Label Distribution")
plt.show()

In [ ]:
train.describe().T

In [ ]:
corr = train.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
train["comment_length"] = train["comment"].str.len()
train["word_count"] = train["comment"].str.split().str.len()

header("TEXT STATISTICS BY LABEL")
print(train.groupby("label")[["comment_length", "word_count"]].mean())

In [ ]:
cat_features = train.select_dtypes(include="object" if IS_KAGGLE else "str").columns
cat_features = [col for col in cat_features if col not in ["comment", "created_date"]]

header("Categorical Features")
for col in cat_features:
    print(f"\n{col} Value counts:")
    print(train[col].value_counts())
    print(f"\n{col} Proportions:")
    print(train[col].value_counts(normalize=True))
    print("\n" + "-" * 50)

In [ ]:
header("Boxplots of Numerical Features to Identify Outliers")

num_cols = train.select_dtypes(include="number").columns.tolist()

for i in range(0, len(num_cols), 4):
    batch = num_cols[i : i + 4]
    fig, axes = plt.subplots(1, len(batch), figsize=(16, 4))
    axes = [axes] if len(batch) == 1 else axes

    for ax, col in zip(axes, batch):
        sns.boxplot(y=train[col], ax=ax, showfliers=False)
        sns.stripplot(y=train[col], ax=ax, alpha=0.3, size=2, color="orange")
        ax.set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
header("Numerical Feature Distributions")

num_cols = train.select_dtypes(include="number").columns.tolist()

for i in range(0, len(num_cols), 4):
    batch = num_cols[i : i + 4]
    fig, axes = plt.subplots(1, len(batch), figsize=(16, 4))
    axes = [axes] if len(batch) == 1 else axes

    for ax, col in zip(axes, batch):
        sns.histplot(train[col], ax=ax, kde=True, bins=30)
        ax.set_title(col)

    plt.tight_layout()
    plt.show()

## Feature Engineering

In [ ]:
def create_features(df):
    df = df.copy()

    # Imputation
    for col in ["race", "religion", "gender"]:
        df[col] = df[col].fillna("none").astype(str).str.lower()

    # Text features
    df["comment_length"] = df["comment"].str.len()
    df["word_count"] = df["comment"].str.split().str.len()
    df["avg_word_length"] = df["comment"].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
    )
    df["unique_word_count"] = df["comment"].apply(
        lambda x: len(set(str(x).lower().split()))
    )
    df["unique_word_ratio"] = df["unique_word_count"] / (df["word_count"] + 1)

    # Special characters, punctuation and intensity
    df["exclamation_count"] = df["comment"].str.count("!")
    df["question_count"] = df["comment"].str.count(r"\?")
    df["capital_ratio"] = df["comment"].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )
    df["digit_count"] = df["comment"].str.count(r"\d")
    df["special_char_count"] = df["comment"].str.count(r"[^a-zA-Z0-9\s]")
    df["uppercase_word_count"] = df["comment"].apply(
        lambda x: sum(1 for w in str(x).split() if w.isupper() and len(w) > 1)
    )
    df["punctuation_ratio"] = df["special_char_count"] / (df["comment_length"] + 1)
    df["repeat_exclaim"] = df["comment"].str.count(r"!{2,}")
    df["repeat_question"] = df["comment"].str.count(r"\?{2,}")
    df["elongated_word_count"] = df["comment"].str.count(r"\b\w*(\w)\1{2,}\w*\b")

    # Lightweight lexicon features
    profanity_terms = [
        "idiot",
        "stupid",
        "dumb",
        "hate",
        "kill",
        "moron",
        "trash",
        "rubish",
    ]
    identity_terms = [
        "black",
        "white",
        "asian",
        "latino",
        "hindu",
        "muslim",
        "christian",
        "jewish",
        "female",
        "male",
        "transgender",
        "lgbtq",
    ]
    profanity_pattern = r"\b(?:" + "|".join(profanity_terms) + r")\b"
    identity_pattern = r"\b(?:" + "|".join(identity_terms) + r")\b"
    df["profanity_count"] = df["comment"].str.lower().str.count(profanity_pattern)
    df["identity_term_count"] = df["comment"].str.lower().str.count(identity_pattern)
    df["has_profanity"] = (df["profanity_count"] > 0).astype(int)

    # Engagement features
    df["total_votes"] = df["upvote"] + df["downvote"]
    df["vote_ratio"] = np.where(
        df["total_votes"] > 0, df["upvote"] / df["total_votes"], 0.5
    )
    df["vote_diff"] = df["upvote"] - df["downvote"]
    df["log_upvote"] = np.log1p(df["upvote"])
    df["log_downvote"] = np.log1p(df["downvote"])
    df["vote_product"] = np.log1p(df["upvote"] * df["downvote"])
    df["controversy_score"] = np.where(
        df["total_votes"] > 0,
        1 - np.abs(df["upvote"] - df["downvote"]) / df["total_votes"],
        0,
    )

    # Length-normalized engagement
    df["downvote_per_word"] = df["downvote"] / (df["word_count"] + 1)
    df["upvote_per_word"] = df["upvote"] / (df["word_count"] + 1)
    df["vote_diff_per_100_chars"] = 100 * df["vote_diff"] / (df["comment_length"] + 1)

    # Temporal features
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["hour"] = df["created_date"].dt.hour
    df["day_of_week"] = df["created_date"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 6)).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    # Emoticon features
    df["total_emoticons"] = df["emoticon_1"] + df["emoticon_2"] + df["emoticon_3"]
    df["has_emoticon"] = (df["total_emoticons"] > 0).astype(int)
    df["emoticon_ratio"] = df["total_emoticons"] / (df["word_count"] + 1)

    # Convert boolean features to integers
    bool_features = df.select_dtypes(include="bool").columns
    df[bool_features] = df[bool_features].astype(int)

    # Demographics + pairwise interactions
    for col in ["race", "religion", "gender"]:
        vc = df[col].value_counts(normalize=True)
        rare_values = vc[vc < 0.01].index
        df[col] = df[col].where(~df[col].isin(rare_values), "other_rare")

    df["race_x_religion"] = df["race"] + "_" + df["religion"]
    df["race_x_gender"] = df["race"] + "_" + df["gender"]
    df["religion_x_gender"] = df["religion"] + "_" + df["gender"]
    df["demographics"] = df["race"] + "_" + df["religion"] + "_" + df["gender"]

    # Indicator combinations
    df["indicator_sum"] = (
        (df["race"] != "none").astype(int)
        + (df["religion"] != "none").astype(int)
        + (df["gender"] != "none").astype(int)
        + df["disability"]
    )
    df["if_sum"] = df["if_1"] + df["if_2"]
    df["if_product"] = df["if_1"] * df["if_2"]

    # Post-level aggregations + rank signals
    post_stats = (
        df.groupby("post_id")
        .agg(
            post_avg_upvote=("upvote", "mean"),
            post_count=("upvote", "count"),
            post_avg_downvote=("downvote", "mean"),
            post_std_upvote=("upvote", "std"),
            post_std_downvote=("downvote", "std"),
        )
        .reset_index()
    )
    post_stats = post_stats.fillna(0)

    df = df.merge(post_stats, on="post_id", how="left")
    df["upvote_vs_post"] = df["upvote"] - df["post_avg_upvote"]
    df["downvote_vs_post"] = df["downvote"] - df["post_avg_downvote"]
    df["upvote_zscore"] = df["upvote_vs_post"] / (df["post_std_upvote"] + 1e-8)
    df["downvote_zscore"] = df["downvote_vs_post"] / (df["post_std_downvote"] + 1e-8)

    df["upvote_rank_pct"] = df.groupby("post_id")["upvote"].rank(pct=True)
    df["downvote_rank_pct"] = df.groupby("post_id")["downvote"].rank(pct=True)
    df["vote_diff_rank_pct"] = df.groupby("post_id")["vote_diff"].rank(pct=True)
    df["is_high_activity_post"] = (df["post_count"] >= 8).astype(int)

    # Text x engagement interactions
    df["length_x_downvote"] = df["comment_length"] * df["log_downvote"]
    df["words_x_downvote"] = df["word_count"] * df["log_downvote"]

    # Keep features numerically stable
    for col in ["vote_diff_per_100_chars", "upvote_zscore", "downvote_zscore"]:
        df[col] = np.clip(df[col], -50, 50)

    # Drop noisy features
    df.drop(columns=["created_date", "post_id"], inplace=True)

    return df

In [ ]:
def add_target_encoding(train_df, test_df, cols, target, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    train_df = train_df.copy()
    test_df = test_df.copy()

    classes = sorted(train_df[target].unique())
    for col in cols:
        for cls in classes:
            col_name = f"{col}_te_{cls}"
            train_df[col_name] = 0.0
            te_test = np.zeros(len(test_df))
            for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
                binary_target = (train_df.iloc[tr_idx][target] == cls).astype(float)

                means = binary_target.groupby(train_df.iloc[tr_idx][col]).mean()
                global_mean = binary_target.mean()

                train_df.iloc[val_idx, train_df.columns.get_loc(col_name)] = (
                    train_df.iloc[val_idx][col].map(means).fillna(global_mean)
                )
                te_test += test_df[col].map(means).fillna(global_mean).values

            test_df[col_name] = te_test / n_splits

    return train_df, test_df

In [ ]:
train, test = add_target_encoding(
    train,
    test,
    ["race", "religion", "gender"],
    "label",
)

In [ ]:
train = create_features(train)
test = create_features(test)

In [ ]:
header("Numerical Feature Distributions After Feature Engineering")

num_cols = train.select_dtypes(include="number").columns.tolist()

for i in range(0, len(num_cols), 4):
    batch = num_cols[i : i + 4]
    fig, axes = plt.subplots(1, len(batch), figsize=(16, 4))
    axes = [axes] if len(batch) == 1 else axes

    for ax, col in zip(axes, batch):
        sns.histplot(train[col], ax=ax, kde=True, bins=30)
        ax.set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
cat_features = train.select_dtypes(
    include=("object" if IS_KAGGLE else "str")
).columns.tolist()
cat_features.remove("comment")
cat_features

## Preprocessing

In [ ]:
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="none")),
        (
            "onehot",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

transformer = ColumnTransformer(
    [("cat_process", cat_pipeline, cat_features)],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# Separate label before transformation (test doesn't have label column)
y_labels = train["label"].copy()
train.drop(columns=["label"], inplace=True)

train = transformer.fit_transform(train)
test = transformer.transform(test)

# Add label back to train
train["label"] = y_labels.values

In [ ]:
header("Post-Preprocessing Dataset Info")
print(f"Total Features: {len(train.columns)}")
print(f"Output Columns: {train.columns.tolist()}")

## Model Building

In [ ]:
train["comment_clean"] = train["comment"].apply(clean_text)
test["comment_clean"] = test["comment"].apply(clean_text)

In [ ]:
# Word-level TF-IDF (Higher max_features + trigrams)
tfidf_word = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 3),
    min_df=15,
    max_df=0.90,
    sublinear_tf=True,
    dtype=np.float32,
    stop_words="english",
)

# Character-level TF-IDF (Captures subword patterns/misspellings)
tfidf_char = TfidfVectorizer(
    max_features=40_000,
    ngram_range=(3, 4),
    min_df=15,
    max_df=0.90,
    sublinear_tf=True,
    analyzer="char_wb",
    dtype=np.float32,
)

In [ ]:
tfidf_word_train = tfidf_word.fit_transform(train["comment_clean"])
tfidf_word_test = tfidf_word.transform(test["comment_clean"])

tfidf_char_train = tfidf_char.fit_transform(train["comment_clean"])
tfidf_char_test = tfidf_char.transform(test["comment_clean"])

print(f"Word TF-IDF: {tfidf_word_train.shape}")
print(f"Char TF-IDF: {tfidf_char_train.shape}")
gc.collect()

In [ ]:
train.drop(columns=["comment", "comment_clean"], inplace=True)
test.drop(columns=["comment", "comment_clean"], inplace=True)

In [ ]:
X_text_train = hstack([tfidf_word_train, tfidf_char_train]).tocsr()
X_text_test = hstack([tfidf_word_test, tfidf_char_test]).tocsr()

if ENABLE_SVD:
    header("Applying TruncatedSVD on text features")
    svd = TruncatedSVD(
        n_components=SVD_N_COMPONENTS,
        n_iter=7,
        random_state=RANDOM_STATE,
    )
    X_text_train = csr_matrix(svd.fit_transform(X_text_train).astype(np.float32))
    X_text_test = csr_matrix(svd.transform(X_text_test).astype(np.float32))
    print(f"SVD explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")
    print(f"SVD text shape (train): {X_text_train.shape}")
else:
    print("SVD disabled; using raw sparse TF-IDF text features.")

In [ ]:
feature_cols = train.select_dtypes(include="number").columns.tolist()

# remove label from features
feature_cols.remove("label")
num_cols = [c for c in feature_cols if c not in cat_features]

In [ ]:
train[num_cols] = train[num_cols].astype("float32")
test[num_cols] = test[num_cols].astype("float32")
train[cat_features] = train[cat_features].astype("int32")
test[cat_features] = test[cat_features].astype("int32")

In [ ]:
X_tab_num = train[num_cols].fillna(0)
X_tab_num_test = test[num_cols].fillna(0)
X_tab_cat = train[cat_features].fillna(-1)
X_tab_cat_test = test[cat_features].fillna(-1)

In [ ]:
# separate columns with enough variance for PowerTransformer
low_variance_cols = [c for c in X_tab_num.columns if X_tab_num[c].nunique() <= 5]
high_variance_cols = [c for c in X_tab_num.columns if c not in low_variance_cols]

# clip extreme values
clip_cols = high_variance_cols
X_tab_num[clip_cols] = X_tab_num[clip_cols].clip(
    lower=X_tab_num[clip_cols].quantile(0.001),
    upper=X_tab_num[clip_cols].quantile(0.999),
    axis=1,
)

# apply PowerTransformer to handle skewness only on high variance columns
pt = PowerTransformer(method="yeo-johnson")
X_tab_num[high_variance_cols] = pt.fit_transform(X_tab_num[high_variance_cols])
X_tab_num_test[high_variance_cols] = pt.transform(X_tab_num_test[high_variance_cols])

# scale all numeric features
scaler = RobustScaler().set_output(transform="pandas")
X_tab_num_scaled = scaler.fit_transform(X_tab_num)
X_tab_num_test_scaled = scaler.transform(X_tab_num_test)

In [ ]:
header("Numerical Feature Distributions After Scaling")

num_cols = X_tab_num_scaled.select_dtypes(include="number").columns.tolist()

for i in range(0, len(num_cols), 4):
    batch = num_cols[i : i + 4]
    fig, axes = plt.subplots(1, len(batch), figsize=(16, 4))
    axes = [axes] if len(batch) == 1 else axes

    for ax, col in zip(axes, batch):
        sns.histplot(X_tab_num_scaled[col], ax=ax, kde=True, bins=30)
        ax.set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
# Combine all features: text branch (raw TF-IDF or SVD) + numeric + categorical
X = hstack([X_text_train, csr_matrix(X_tab_num_scaled), csr_matrix(X_tab_cat)]).tocsr()
X_sub = hstack(
    [X_text_test, csr_matrix(X_tab_num_test_scaled), csr_matrix(X_tab_cat_test)]
).tocsr()
y = train["label"].values

cat_indices = list(range(X.shape[1] - len(cat_features), X.shape[1]))

print(f"Combined training shape: {X.shape}")
print(f"Combined test shape: {X_sub.shape}")
gc.collect()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [ ]:
header("Sparse Matrix Stats")

print("Train Matrix:")
print(f"\tShape: {X.shape}")
print(f"\tSparsity: {calc_sparsity(X):.2f}%")
print(f"\tSparse size: {X.data.nbytes / 1e6:.2f} MB")

print("\nSubmission Matrix:")
print(f"\tShape: {X_sub.shape}")
print(f"\tSparsity: {calc_sparsity(X_sub):.2f}%")
print(f"\tSparse size: {X_sub.data.nbytes / 1e6:.2f} MB")

# Cleanup
gc.collect()

In [ ]:
models = {
    # Gradient Boosting Models
    "XGBoost": XGBClassifier(
        n_estimators=1500,
        max_depth=6,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.6,
        min_child_weight=3,
        random_state=RANDOM_STATE,
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=1500,
        max_depth=-1,
        num_leaves=63,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=30,
        objective="multiclass",
        random_state=RANDOM_STATE,
        verbose=-1,
        class_weight="balanced",
        n_jobs=-1,
    ),
    # "HistGradientBoosting": HistGradientBoostingClassifier(
    #     max_iter=1000,
    #     learning_rate=0.05,
    #     max_depth=8,
    #     l2_regularization=0.1,
    #     class_weight="balanced",
    #     early_stopping=True,
    #     random_state=RANDOM_STATE,
    # ),
    # Tree-Based Ensemble Models
    # "RandomForest": RandomForestClassifier(
    #     n_estimators=300,
    #     max_depth=10,
    #     random_state=RANDOM_STATE,
    #     class_weight="balanced",
    #     n_jobs=-1,
    # ),
    # "ExtraTrees": ExtraTreesClassifier(
    #     n_estimators=300,
    #     max_depth=10,
    #     random_state=RANDOM_STATE,
    #     class_weight="balanced",
    #     n_jobs=-1,
    # ),
    # Linear Models
    "LogisticRegression": LogisticRegression(
        C=10.0,
        max_iter=2000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
    ),
    "LinearSVC_Calibrated": CalibratedClassifierCV(
        estimator=LinearSVC(
            C=1.0,
            dual=False,
            class_weight="balanced",
            max_iter=4000,
            random_state=RANDOM_STATE,
        ),
        method="sigmoid",
        cv=2,
    ),
    # Neural Network
    # "MLP": MLPClassifier(
    #     hidden_layer_sizes=(256, 128, 64),
    #     activation="relu",
    #     solver="adam",
    #     alpha=0.0001,
    #     batch_size=256,
    #     learning_rate="adaptive",
    #     max_iter=500,
    #     random_state=RANDOM_STATE,
    #     early_stopping=True,
    # ),
}

In [ ]:
scores = {}

for model_name, model in models.items():
    header(f"Training {model_name}")
    if not ENABLE_XGBOOST and "XGBoost" in model_name:
        print(f"Skipping {model_name} due to long training time.\n")
        continue

    if model_name == "XGBoost":
        sample_weights = compute_sample_weight("balanced", y_train)
        model.fit(X_train, y_train, sample_weight=sample_weights)
    elif model_name == "LightGBM":
        model.fit(X_train, y_train, categorical_feature=cat_indices)
    else:
        model.fit(X_train, y_train)

    val_preds = model.predict(X_test)

    score = f1_score(y_test, val_preds, average="macro")
    scores[model_name] = {
        "model": model,
        "score": score,
    }

    print(f"\t{model_name} Macro F1 Score: {score:.4f}")

In [ ]:
# sort scores
scores = list(sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True))

## Hyperparameter Tuning

In [ ]:
params_dict = {
    # Gradient Boosting Models
    "XGBoost": {
        "n_estimators": [200, 300, 500],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_child_weight": [1, 5, 10],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
    },
    "LightGBM": {
        "n_estimators": [200, 300, 500],
        "num_leaves": [31, 63, 127],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_child_samples": [10, 20, 50],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "class_weight": ["balanced", None],
    },
    "HistGradientBoosting": {
        "max_iter": [200, 300, 500],
        "max_depth": [5, 7, 10, None],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_samples_leaf": [10, 20, 50],
        "l2_regularization": [0, 0.1, 1],
        "class_weight": ["balanced", None],
    },
    # Tree-Based Ensemble Models
    "RandomForest": {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.3],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    "ExtraTrees": {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.3],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    # Linear Models
    "LogisticRegression": {
        "C": [0.01, 0.1, 1, 10],
        "class_weight": ["balanced", None],
    },
    # Neural Network
    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64), (128, 64, 32)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.01],
        "batch_size": [64, 128, 256],
    },
}

In [ ]:
if PERFORM_TUNING:
    for model_name, model_data in scores[:3]:
        header(f"Hyperparameter Tuning for {model_name}")

        fit_params = {}
        if model_name == "XGBoost":
            weights = compute_sample_weight("balanced", y_train)
            fit_params["sample_weight"] = weights

        search_cv = RandomizedSearchCV(
            estimator=models[model_name],
            param_distributions=params_dict[model_name],
            n_iter=5,
            scoring="f1_macro",
            cv=3,
            verbose=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        search_cv.fit(X_train, y_train, **fit_params)
        scores.append(
            (
                f"{model_name}_Tuned",
                {
                    "score": search_cv.best_score_,
                    "model": search_cv.best_estimator_,
                },
            )
        )

        print(f"\t{model_name}_Tuned Best Score: {search_cv.best_score_:.4f}")
else:
    print("Skipping hyperparameter tuning (PERFORM_TUNING = False).")
    print("Using optimized default parameters instead.")

In [ ]:
# sort scores
scores = list(sorted(scores, key=lambda x: x[1]["score"], reverse=True))

## Ensemble Models

In [ ]:
classes = np.unique(y)

n_models = min(len(scores), TOP_K)
model_preds = []
model_names = []
for model_name, model_data in scores[:n_models]:
    header(f"Ensemble Model: {model_name}")
    print(f"Macro F1 Score: {model_data['score']:.4f}")
    model = model_data["model"]
    model_preds.append(model.predict_proba(X_test))
    model_names.append(model_name)
    classes = model.classes_

model_preds = np.array(model_preds)

header("Finding optimal weights")


def objective_func(weights):
    w = np.exp(weights) / np.sum(np.exp(weights))
    weighted_probs = np.tensordot(w, model_preds, axes=(0, 0))
    preds = classes[np.argmax(weighted_probs, axis=1)]
    return -f1_score(y_test, preds, average="macro")


initial_weights = np.zeros(n_models)
result = minimize(objective_func, initial_weights, method="Nelder-Mead")
optimal_weights = np.exp(result.x) / np.sum(np.exp(result.x))

optimal_weights_dict = {name: w for name, w in zip(model_names, optimal_weights)}

print(f"\nOptimal weights found:")
for name, w in optimal_weights_dict.items():
    print(f"  {name}: {w:.4f}")

test_probs = np.tensordot(optimal_weights, model_preds, axes=(0, 0))

In [ ]:
header("Ensemble Model Results")
ensemble_pred_indices = np.argmax(test_probs, axis=1)
ensemble_preds = classes[ensemble_pred_indices]

ensemble_model_name = f"Ensemble_Top_{TOP_K}"
ensemble_score = f1_score(y_test, ensemble_preds, average="macro")

scores.append(
    (
        ensemble_model_name,
        {
            "score": ensemble_score,
            "model": None,
        },
    )
)

print(f"Ensemble Macro F1 Score: {ensemble_score:.4f}")

print("\nEnsemble Classification Report:")
print(classification_report(y_test, ensemble_preds))

## Model Evaluation

In [ ]:
scores = list(sorted(scores, key=lambda x: x[1]["score"], reverse=True))

header("Final Leaderboard")
print(f"{'Rank':<5} {'Model Name':<25} {'Score':<10}")
print("-" * 65)

for rank, (name, data) in enumerate(scores, 1):
    print(f"{rank:<5} {name:<25} {data['score']:.4f}")

# Update best_model_data to the new #1
best_model_name, best_model_data = scores[0]
print(
    f"\n>> New Best Model: {best_model_name} with Score: {best_model_data['score']:.4f}"
)

## Predicting Test Data

In [ ]:
if best_model_name == ensemble_model_name:
    # Use ensemble model
    final_probs = np.zeros((X_sub.shape[0], len(np.unique(y))))
    print("Generating Ensemble submission...")
    header("Retraining Top Models on Full Data")

    selected_models = [m for m in scores if m[0] in optimal_weights_dict]

    for model_name, model_data in selected_models:
        model_weight = optimal_weights_dict[model_name]
        # TODO: Implement this properly
        # if not ENABLE_XGBOOST_RETRAINING and "XGBoost" in model_name:
        #     print(f"Skipping retraining for {model_name} due to long training time.\n")
        #     continue

        print(f"Retraining {model_name}...")
        print(f"Macro F1 Score: {model_data['score']:.4f}")

        print("Retraining on full training data...")
        model = model_data["model"]

        if "XGBoost" in model_name:
            sample_weights = compute_sample_weight("balanced", y)
            model.fit(X, y, sample_weight=sample_weights)
        elif "LightGBM" in model_name:
            model.fit(X, y, categorical_feature=cat_indices)
        else:
            model.fit(X, y)

        # Final predictions
        model_preds = model.predict_proba(X_sub)
        final_probs += model_preds * model_weight

        print(f"{model_name} retrained on full data...\n")
else:
    # Use single best model
    print(
        f"Ensemble score ({ensemble_score:.4f}) did not beat single best model ({best_model_data['score']:.4f})."
    )

    header(f"Best Model: {best_model_name}")
    print(f"Macro F1 Score: {best_model_data['score']:.4f}")

    print("Classification Report:")
    best_val_preds = best_model_data["model"].predict(X_test)
    print(classification_report(y_test, best_val_preds))

    print("\nRetraining on full training data...")
    model = best_model_data["model"]

    if "XGBoost" in best_model_name:
        sample_weights = compute_sample_weight("balanced", y)
        model.fit(X, y, sample_weight=sample_weights)
    elif "LightGBM" in best_model_name:
        model.fit(X, y, categorical_feature=cat_indices)
    else:
        model.fit(X, y)

    final_probs = model.predict_proba(X_sub)

# Final predictions
final_pred_indices = np.argmax(final_probs, axis=1)
final_preds = classes[final_pred_indices]
print("Final submission generated.")

## Final Submission

In [ ]:
submission = pd.DataFrame({"ID": test.index + 1, "label": final_preds})
submission.to_csv("submission.csv", index=False)

header("Submission created!")
print(submission["label"].value_counts())

--- 
**⚠️ Automated Merge Info**

The cells below were added automatically by the merge script to control execution flow.

In [ ]:
# stop execution here

import sys
sys.exit(0)

# Milestones

## Milestone 1: Data loading and preprocessing

**Deadline:** `Feb 15, 2026`

- Data loading
- Exploratory Data Analysis
    - Data visualization
    - Statistical analysis
- Train-Validation Split
- Data Cleaning/ Preprocessing
- Building a baseline model.

### Import libraries

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

### Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

### Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

### Question 1

What is the shape of the training dataset?

- [ ] (102000, 15)
- [ ] (198000, 14)
- [x] (198000, 15)
- [ ] (300000, 15)

In [4]:
train.shape

(198000, 15)

### Question 2

How many columns are present in the test dataset?

In [5]:
test.shape[1]

14

### Question 3

How many columns in the training dataset have **object** data type?

In [6]:
train.select_dtypes(include="object").shape[1]

C:\Users\varun\AppData\Local\Temp\ipykernel_9608\1487577859.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  train.select_dtypes(include="object").shape[1]


5

### Question 4

How many **numerical** columns are present in the training dataset?

In [7]:
train.select_dtypes(include="number").shape[1]

9

### Question 5

Which of the following columns is of boolean type?

- [ ] gender
- [ ] race
- [x] disability
- [ ] emoticon_1

In [8]:
train.select_dtypes(include="bool").columns

Index(['disability'], dtype='str')

### Question 6

Which of the following columns have missing values?

- [x] race
- [x] religion
- [x] gender
- [x] comment

In [9]:
train.columns[train.isna().sum() > 0]

Index(['race', 'religion', 'gender', 'comment'], dtype='str')

### Question 7

How many distinct target classes are present in the dataset?

In [10]:
train["label"].nunique()

4

### Question 8

What percentage of the dataset approximately belongs to label 0?

- [ ] ~30%
- [ ] ~45%
- [x] ~58%
- [ ] ~75%

In [11]:
np.round(train["label"].value_counts(normalize=True) * 100)  # type: ignore[unsupported-operator]

label
0    58.0
2    32.0
1     8.0
3     3.0
Name: proportion, dtype: float64

### Question 9

What is the median number of upvotes per comment?

- [ ] 0
- [x] 1
- [ ] 2
- [ ] 3

In [12]:
train["upvote"].quantile(0.5)

np.float64(1.0)

### Question 10

Which numerical feature shows the largest maximum value?

- [ ] upvote
- [ ] downvote
- [x] if_1
- [ ] if_2

In [13]:
train.select_dtypes(include="number").max().sort_values(ascending=False)

if_1          1860
if_2          1833
upvote         201
post_id        129
downvote       107
emoticon_1      47
emoticon_3      17
emoticon_2      11
label            3
dtype: int64

### Question 11

What is the minimum value of `if_2`?

In [14]:
train["if_2"].min()

np.int64(3)

## Milestone 2: Model Training and Hyperparameter Tuning

**Deadline:** `Feb 22, 2026`

1. [Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)
2. [Stochastic Gradient Descent](https://scikit-learn.org/stable/modules/sgd.html)

Tune the hyperparameters of the model.

### Import libraries

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

### Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

### Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

### Question 1

In the context of text preprocessing for NLP, which condition must be checked in addition to NaN values when assessing missingness in the comment column of train.csv?

- [ ] Presence of duplicate comments
- [ ] Presence of out-of-vocabulary tokens
- [x] Presence of empty or whitespace-only strings
- [ ] Presence of rare words

### Question 2

Convert the created_date column into datetime objects. Identify the month that occurs the most number of times in the dataset and enter the answer as a lowercase string.

In [4]:
train["created_date"] = pd.to_datetime(train["created_date"])
most_common_month = train["created_date"].dt.month.mode()[0]
pd.to_datetime(most_common_month, format="%m").strftime("%B").lower()

'may'

### Question 3

Create a new feature called total_emoticons by calculating the sum of `emoticon_1`, `emoticon_2`, and `emoticon_3` for each row. What is the maximum value observed in this new feature across the entire dataset?

In [5]:
train["total_emoticons"] = (
    train["emoticon_1"] + train["emoticon_2"] + train["emoticon_3"]
)

train["total_emoticons"].max()

np.int64(60)

### Question 4

Calculate the median character length (including spaces) of the comment column for all entries where label is equal to 3. (**Note:** Treat any missing comments as empty strings).

In [6]:
train["comment"] = train["comment"].fillna("")
train[train["label"] == 3]["comment"].str.len().median()

np.float64(128.0)

### Question 5

If you apply Min-Max Scaling to the upvote column to normalize it between the range [0, 1], what would be the resulting scaled value for an observation that originally had exactly 10 upvotes?

- [ ] 0.0345
- [ ] 0.0612
- [ ] 0.0521
- [x] 0.0498

In [7]:
scaler = MinMaxScaler()
train["upvote_scaled"] = scaler.fit_transform(train[["upvote"]])

train.loc[train["upvote"] == 10, "upvote_scaled"].iloc[0].round(4)

np.float64(0.0498)

### Question 6

What is the average word count (number of words separated by whitespace) for comments that are assigned a label of 1? (Round your answer to 2 decimal places).

In [8]:
train[train["label"] == 1]["comment"].str.split().str.len().mean().round(2)  # type: ignore[unresolved-attribute]

np.float64(57.23)

### Question 7

How many comments in the entire dataset contain the substring "Trump"? The search should be case-insensitive.

In [9]:
train["comment"].str.contains("Trump", case=False, regex=False).sum()

np.int64(24398)

### Question 8

Take the comment at the very first row (index 0).

- Remove all punctuation.
- Consider the following words as stop words and remove the same :  ['a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to', 'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his', 'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your']

How many words are left in the text after these two operations?

In [10]:
comment = train.loc[0, "comment"]

# fmt: off
stop_words =  [
    'a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to',
    'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have',
    'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his',
    'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your'
]
# fmt: on

len([w for w in re.sub(r"[^\w\s]", "", comment).split() if w.lower() not in stop_words])  # type: ignore[no-matching-overload]

12

### Question 9

Convert all the text in comment column to lowercase and tokenizing using whitespace. Compute the total number of unique tokens in the dataset and enter the value.

In [11]:
train["comment"] = train["comment"].str.lower()
unique_tokens = set()
for comment in train["comment"]:
    unique_tokens.update(comment.split())

len(unique_tokens)

425413

### Question 10

Apply the TfidfVectorizer to the comment column of train.csv with stop_words as "english", min_df as 5 and ngram_range as (1,2).

How many TF-IDF features are generated using this configuration?

In [12]:
tfidf = TfidfVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2))

tfidf.fit(train["comment"])
len(tfidf.get_feature_names_out())

132974

## Milestone 3: Model Improvement

**Deadline:** `Mar 1, 2026`

1. Dimensionality Reduction
2. Feature Selection
3. Retrain the models
    - [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
    - [Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
    - [Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)

### Import libraries

In [1]:
import os
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

### Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

### Question 1

Split the train dataset using train_test_split with random_state=42 such that 80% is training data and remaining 20% is validation data. store the data as X_train, X_val, y_train, y_val. Lets say training data (X_train) has shape (a,b) where a is number of rows and b is the number of features, similarly validation set (X_val) has shape (c,d) where c is the number of rows and d is the number of features. what will be the value of a + c ?

In [4]:
X = train.drop(columns=["post_id", "label"])
y = train["label"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
X_train.shape[0] + X_val.shape[0]

198000

### Question 2

After converting created_date to a datetime object and extracting the day, month, and year for your X_train and X_val created in the first question, identify the most frequently occurring month across all years in X_train. Which month is it?

In [6]:
X_train["created_date"] = pd.to_datetime(X_train["created_date"])
X_val["created_date"] = pd.to_datetime(X_val["created_date"])

X_train["day"] = X_train["created_date"].dt.day
X_train["month"] = X_train["created_date"].dt.month
X_train["year"] = X_train["created_date"].dt.year

X_val["day"] = X_val["created_date"].dt.day
X_val["month"] = X_val["created_date"].dt.month
X_val["year"] = X_val["created_date"].dt.year

X_train["month"].mode()[0]

np.int32(5)

### Question 3

Impute the null values of categorical features with the value 'none'. Encode only `'religion', 'gender', 'race'` features using one hot encoding, setting `handle_unknown='ignore'`. Make sure the output is a pandas dataframe. Let's say the shape of X_train after imputing is (a,b), what is the value of b?\
**Note:** make sure to transform X_val as well using one hot encoding.

In [7]:
cat_features = X_train.select_dtypes(include="str").columns
cat_features

Index(['race', 'religion', 'gender', 'comment'], dtype='str')

In [8]:
X_train[cat_features] = X_train[cat_features].fillna("none")
X_val[cat_features] = X_val[cat_features].fillna("none")

In [9]:
one_hot = ColumnTransformer(
    [
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["religion", "gender", "race"],
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_train = one_hot.fit_transform(X_train)
X_val = one_hot.transform(X_val)

In [10]:
X_train.shape[1]

32

### Question 4

Apply **CountVectorizer** to the column **comment** of X_train and X_val obtained in previous questions, what is the sum of the counts for the document at index 1 (the second row) of the transformed X_train sparse matrix?\
**Note:** Please ensure to transform X_val as well.

In [11]:
count_vec = CountVectorizer()

X_train_count = count_vec.fit_transform(X_train["comment"])
X_val_count = count_vec.transform(X_val["comment"])

In [12]:
X_train_count[1].sum()

np.int64(41)

### Question 5

Convert **'disability'** feature into integer type, with True being mapped to 1 and False mapped to 0. What is the sum of all the 'disability values in X_train and X_val after the transformation?

In [13]:
X_train["disability"] = X_train["disability"].map({True: 1, False: 0})
X_val["disability"] = X_val["disability"].map({True: 1, False: 0})

In [14]:
X_train["disability"].sum() + X_val["disability"].sum()

np.int64(2743)

### Question 6

Scale the numeric features using **StandardScaler**. What are the number of features seen during fit for X_train? Ensure that you drop all **datetime** columns first.

In [15]:
dt_cols = X_train.select_dtypes(
    include=["datetime64[ns]", "datetime64[ns, UTC]"]
).columns
X_train_no_dt = X_train.drop(columns=dt_cols)

num_features = X_train_no_dt.select_dtypes(include="number").columns
scaler = StandardScaler()
scaler.fit(X_train_no_dt[num_features])

scaler.n_features_in_  # type: ignore

30

### Question 7

Preprocess the data in the following manner and train the model:

Use the full raw data and first split it into training and validation sets so that 80% of the data forms **X_train, y_train** and the remaining 20% forms **X_val, y_val**.

**After splitting:**

1. Impute missing values **by fitting the SimpleImputer on X_train**. You are free to use any valid strategy for Imputation.
2. Use the fitted imputer to transform both **X_train and X_val**
3. Ensure that after imputation no value in X_train or X_val is negative.
4. If any negative value exists, convert it to positive using the absolute value function
5. Convert the created_date feature into a **datetime** object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.
6. Transform the "comment" feature using **TfidfVectorizer** with stop_words='english'. Make sure to transform X_val accordingly.
7. Encode all the categorical features using one hot encoder with handle_unknown='ignore'. Make sure to transform X_val accordingly.
8. Now train a Multinomial Naive Bayes model on the preprocessed data. Use a pipeline if you need to.

What is the macro F1 score obtained on the train dataset?

**Note:** You may train Multinomial Naive Bayes model on the preprocessed data which has been obtained from previous questions and give the answer, both ways the answers will be accepted.

If some feature is creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).

In [16]:
def preprocess(df):
    df = df.copy()
    df["day"] = df["created_date"].dt.day
    df["month"] = df["created_date"].dt.month
    df["year"] = df["created_date"].dt.year
    df.drop(columns=["created_date"], inplace=True)

    num_features = df.select_dtypes(include="number").columns
    df[num_features] = df[num_features].abs()
    return df

In [17]:
X_train_proc = preprocess(X_train)
X_val_proc = preprocess(X_val)

In [18]:
preprocessor = ColumnTransformer(
    transformers=[("tfidf", TfidfVectorizer(stop_words="english"), "comment")],
    remainder="passthrough",
)

pipeline = Pipeline([("preprocessor", preprocessor), ("clf", MultinomialNB())])

pipeline.fit(X_train_proc, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('tfidf', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [19]:
y_train_pred = pipeline.predict(X_train_proc)
f1_score(y_train, y_train_pred, average="macro")

0.4841311504632059

### Question 8

What is the macro F1 score obtained on the validation dataset?

In [20]:
y_val_pred = pipeline.predict(X_val_proc)
f1_score(y_val, y_val_pred, average="macro")

0.45926776710921624

### Question 9

Process the data in the following manner as given below and train the model.

Use the full raw data and first split it into training and validation sets so that 80% of the data forms X_train, y_train and the remaining 20% forms **X_val, y_val**.

After splitting:
1. Impute missing values **by fitting the SimpleImputer on X_train**. You are free to use any valid strategy for Imputation.
2. Use the fitted imputer to transform both **X_train and X_val**
3. Ensure that after imputation no value in X_train or X_val is negative.
4. If any negative value exists, convert it to positive using the absolute value function
5. Convert the created_date feature into a **datetime** object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.
6. Create a new binary categorical column named **is_weekend** where Saturday and Sunday are represented as **1** and all other days as **0**.
7. Apply **TfidfVectorizer(stop_words='english')** to the comment column. Make sure to transform X_val accordingly.
8. Apply **OneHotEncoder(handle_unknown='ignore')** to the categorical features including newly created categorical feature **is_weekend**. Make sure to transform X_val accordingly.

After all above transformation, train **MultinomialNB** model.

After training, use the model to predict the labels for X_train and calculate the **Macro F1 score**. What is the resulting score rounded to four decimal places?

**Note:** If some features are creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).

In [21]:
def preprocess_v2(df):
    df = df.copy()
    df["day"] = df["created_date"].dt.day
    df["month"] = df["created_date"].dt.month
    df["year"] = df["created_date"].dt.year
    df["is_weekend"] = df["created_date"].dt.dayofweek.apply(
        lambda x: 1 if x >= 5 else 0
    )
    df.drop(columns=["created_date"], inplace=True)

    num_features = df.select_dtypes(include="number").columns
    df[num_features] = df[num_features].abs()
    return df

In [22]:
X_train_proc_v2 = preprocess_v2(X_train)
X_val_proc_v2 = preprocess_v2(X_val)

In [23]:
preprocessor = ColumnTransformer(
    [
        ("tfidf", TfidfVectorizer(stop_words="english"), "comment"),
        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["is_weekend"]),
    ],
    remainder="passthrough",
)

pipeline = Pipeline([("preprocessor", preprocessor), ("clf", MultinomialNB())])

pipeline.fit(X_train_proc_v2, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('tfidf', ...), ('ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [24]:
y_train_pred = pipeline.predict(X_train_proc_v2)
f1_score(y_train, y_train_pred, average="macro")

0.4840696062885761

### Question 10

What is the macro F1-score on validation set for the previous question's setup?

In [25]:
y_val_pred = pipeline.predict(X_val_proc_v2)
f1_score(y_val, y_val_pred, average="macro")

0.4592333361559341